In [ ]:
# Package requirements
# pip install transformers torch
# !pip install datasets

In [12]:
import pandas as pd
import re

import pandas as pd
from datasets import Dataset, load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from transformers import pipeline


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Load datasets
master_data = pd.read_csv('/content/drive/MyDrive/AA USD/AAI 590 Capstone/Data/AAPL_MasterData- 2018-2023.csv', parse_dates=['Date'])
article_data = pd.read_csv('/content/drive/MyDrive/AA USD/AAI 590 Capstone/Data/apple_articles_with_sentiment.csv', parse_dates=['date'])

# Rename columns for clarity and consistency
master_data.rename(columns={'AAPL_Close': 'Close'}, inplace=True)
article_data.rename(columns={'date': 'Date', 'article_content': 'Content', 'sentiment_score': 'SentimentScore', 'sentiment': 'Sentiment'}, inplace=True)

# Merge datasets on the date
combined_data = pd.merge(master_data, article_data, on='Date', how='outer')

# Fill or interpolate missing values
combined_data['SentimentScore'].fillna(method='ffill', inplace=True)

# Feature Engineering: Calculate rolling averages for sentiment and stock prices
combined_data['RollingMeanClose'] = combined_data['Close'].rolling(window=7).mean()
combined_data['RollingMeanSentiment'] = combined_data['SentimentScore'].rolling(window=7).mean()


<ipython-input-5-470985accffe>:17: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  combined_data['SentimentScore'].fillna(method='ffill', inplace=True)


In [6]:
combined_data.tail(2)

,Date,Close,Close_Next_1_Day,Close_Next_5_Days,Close_Next_30_Days,Close_Next1_pctchg,Close_Next5_pctchg,Close_Next30_pctchg,composite_sentiment_score,SMA_20,...,title,description,article_url,author,source,Content,SentimentScore,Sentiment,RollingMeanClose,RollingMeanSentiment
9507,2021-12-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Could This $2.6 Trillion Company Still Have Ro...,Don't make the mistake of thinking massive com...,https://www.fool.com/investing/2021/12/04/coul...,"newsfeedback@fool.com (Matthew Frankel, CFP®, ...",The Motley Fool,"With a market cap of more than $2.6 trillion, ...",0.253,Positive,NaN,0.224714
9508,2022-07-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"1 Stock To Buy, 1 Stock To Dump This Week: Ama...",No Description,https://www.investing.com/analysis/1-stock-to-...,Jesse Cohen/Investing.com,Investing.com,"With a market cap of $1.18 trillion, the Seatt...",0.125,Positive,NaN,0.211857


In [7]:
# Create training examples in a conversational format
training_examples = []

for _, row in combined_data.iterrows():
    # Example 1: Sentiment and stock price with content summary
    prompt = f"Tell me the sentiment, stock price, and summarize the news about AAPL on {row['Date'].date()}."

    # Safely handle NaN in the Content column
    if isinstance(row['Content'], str):
        content_summary = " ".join(row['Content'].split()[:30])  # Simple summary: first 30 words
    else:
        content_summary = "No news content available."

    response = (
        f"On {row['Date'].date()}, the sentiment score was {row['SentimentScore']} and the closing stock price was {row['Close']} USD. "
        f"Here's a summary of the news: {content_summary}."
    )
    training_examples.append((prompt, response))

    # Example 2: Financial performance - Gross Margin
    prompt = f"What was AAPL's gross margin on {row['Date'].date()}?"
    response = f"AAPL's gross margin on {row['Date'].date()} was {row['GrossMargin']}%."
    training_examples.append((prompt, response))

    # Example 3: Technical indicators - RSI
    prompt = f"What was the RSI for AAPL on {row['Date'].date()}?"
    response = f"The RSI for AAPL on {row['Date'].date()} was {row['RSI']}."
    training_examples.append((prompt, response))

    # Example 4: Next day's closing price prediction
    prompt = f"What is the predicted closing price for AAPL on the next day after {row['Date'].date()}?"
    response = f"The predicted closing price for AAPL on the next day after {row['Date'].date()} is {row['Close_Next_1_Day']} USD."
    training_examples.append((prompt, response))

    # Example 5: SMA value
    prompt = f"What was the 20-day SMA for AAPL on {row['Date'].date()}?"
    response = f"The 20-day SMA for AAPL on {row['Date'].date()} was {row['SMA_20']}."
    training_examples.append((prompt, response))

    # Example 6: EPS
    prompt = f"What was AAPL's EPS on {row['Date'].date()}?"
    response = f"AAPL's EPS on {row['Date'].date()} was {row['EPS']}."
    training_examples.append((prompt, response))

    # Example 7: Operating cash flow
    prompt = f"What was AAPL's operating cash flow on {row['Date'].date()}?"
    response = f"AAPL's operating cash flow on {row['Date'].date()} was {row['OperatingCashFlow']} USD."
    training_examples.append((prompt, response))

    # Example 8: Quick Ratio
    prompt = f"What was AAPL's quick ratio on {row['Date'].date()}?"
    response = f"AAPL's quick ratio on {row['Date'].date()} was {row['QuickRatio']}."
    training_examples.append((prompt, response))

    # Example 9: Debt to Equity Ratio
    prompt = f"What was AAPL's debt to equity ratio on {row['Date'].date()}?"
    response = f"AAPL's debt to equity ratio on {row['Date'].date()} was {row['DebtToEquityRatio']}."
    training_examples.append((prompt, response))

    # Example 10: Technical indicator - MACD
    prompt = f"What was the MACD value for AAPL on {row['Date'].date()}?"
    response = f"The MACD value for AAPL on {row['Date'].date()} was {row['MACD']}."
    training_examples.append((prompt, response))

    # Example 11: Gross Profit
    prompt = f"What was AAPL's gross profit on {row['Date'].date()}?"
    response = f"AAPL's gross profit on {row['Date'].date()} was {row['GrossProfit']} USD."
    training_examples.append((prompt, response))

    # Example 12: Revenue
    prompt = f"What was AAPL's total revenue on {row['Date'].date()}?"
    response = f"AAPL's total revenue on {row['Date'].date()} was {row['TotalRevenue']} USD."
    training_examples.append((prompt, response))

    # Example 13: Net Income
    prompt = f"What was AAPL's net income on {row['Date'].date()}?"
    response = f"AAPL's net income on {row['Date'].date()} was {row['NetIncome']} USD."
    training_examples.append((prompt, response))

    # Example 14: Current Ratio
    prompt = f"What was AAPL's current ratio on {row['Date'].date()}?"
    response = f"AAPL's current ratio on {row['Date'].date()} was {row['CurrentRatio']}."
    training_examples.append((prompt, response))

    # Example 15: SMA-20
    prompt = f"What was the 20-day SMA for AAPL on {row['Date'].date()}?"
    response = f"The 20-day SMA for AAPL on {row['Date'].date()} was {row['SMA_20']}."
    training_examples.append((prompt, response))

    # Example 16: Sentiment description
    prompt = f"What was the news sentiment about AAPL on {row['Date'].date()}?"
    response = f"On {row['Date'].date()}, the sentiment was described as {row['Sentiment']} with a score of {row['SentimentScore']}."
    training_examples.append((prompt, response))

    # Example 17: EPS (Basic)
    prompt = f"What was AAPL's basic EPS on {row['Date'].date()}?"
    response = f"AAPL's basic EPS on {row['Date'].date()} was {row['BasicEPS']}."
    training_examples.append((prompt, response))

    # Example 18: Change in Inventory
    prompt = f"What was the change in AAPL's inventory on {row['Date'].date()}?"
    response = f"The change in inventory for AAPL on {row['Date'].date()} was {row['ChangeInInventory']} USD."
    training_examples.append((prompt, response))

    # Example 19: Operating Income
    prompt = f"What was AAPL's operating income on {row['Date'].date()}?"
    response = f"AAPL's operating income on {row['Date'].date()} was {row['OperatingIncome']} USD."
    training_examples.append((prompt, response))

    # Example 20: Cash and Cash Equivalents
    prompt = f"What was AAPL's cash and cash equivalents on {row['Date'].date()}?"
    response = f"AAPL's cash and cash equivalents on {row['Date'].date()} was {row['CashAndCashEquivalents']} USD."
    training_examples.append((prompt, response))

# Convert to DataFrame
df_train = pd.DataFrame(training_examples, columns=['prompt', 'completion'])

# Use regex to extract the date from the prompt and split datasets
date_pattern = re.compile(r'on (\d{4}-\d{2}-\d{2})')

# Extract date and handle missing matches
def extract_date(prompt):
    match = date_pattern.search(prompt)
    return match.group(1) if match else None

# Apply the function to extract dates
df_train['date'] = df_train['prompt'].apply(extract_date)

# Drop rows where the date could not be extracted
df_train = df_train.dropna(subset=['date'])

# Convert the date column to datetime
df_train['date'] = pd.to_datetime(df_train['date'])

# Split the data into training and evaluation sets
train_df = df_train[df_train['date'] <= pd.Timestamp('2023-06-30')]
eval_df = df_train[df_train['date'] > pd.Timestamp('2023-06-30')]

# Drop the temporary date column after splitting
train_df = train_df.drop(columns=['date'])
eval_df = eval_df.drop(columns=['date'])


In [8]:
# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

# Load pre-trained model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Add a padding token if not present
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))  # Resize token embeddings for new token


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Embedding(50258, 768)

In [9]:
# Tokenization function
def tokenize_function(examples):
    concatenated_examples = [ex1 + " " + ex2 for ex1, ex2 in zip(examples['prompt'], examples['completion'])]
    return tokenizer(concatenated_examples, truncation=True, padding="max_length", max_length=512)

# Apply tokenization
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Data collator for padding dynamically
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Setup training arguments
training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=32,
    save_steps=500,
    eval_steps=500,
    evaluation_strategy="steps",
    logging_dir='./logs',
    load_best_model_at_end=True
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

# Train the model
trainer.train()

# Save the model
model.save_pretrained('./chatbot_model')

Map:   0%|          | 0/156560 [00:00<?, ? examples/s]

Map:   0%|          | 0/24092 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss
500,1.221900,0.802397
1000,0.582300,0.849021
1500,0.563300,0.862313
2000,0.546100,0.880417
2500,0.530900,0.907160
3000,0.537200,0.909575
3500,0.517800,0.902301
4000,0.514700,0.928210
4500,0.516300,0.921634


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [13]:
# Load the trained model and tokenizer
chatbot = pipeline('text-generation', model=model, tokenizer=tokenizer)

# Sample prompts to test the model
sample_prompts = [
    "Tell me the sentiment and stock price of AAPL on 2022-01-03.",
    "What was the revenue for 2022-01-03?",
    "What was the sentiment on AAPL during April 2022?"
]

# Generate responses for the sample prompts
for prompt in sample_prompts:
    response = chatbot(prompt, max_length=100, num_return_sequences=1)
    print(f"Prompt: {prompt}")
    print(f"Response: {response[0]['generated_text']}\n")


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Prompt: Tell me the sentiment and stock price of AAPL on 2022-01-03.
Response: Tell me the sentiment and stock price of AAPL on 2022-01-03. On 2022-01-03, the sentiment score was 0.11 and the closing stock price was 143.73 USD. Here's a summary of the news: It seems like the world's largest e-commerce company is still under the regulatory influence of an IRS. The company is under the jurisdiction of the IRS, said Apple Technology Solutions Corporation. These companies are still subject to an annual payment which can be described

Prompt: What was the revenue for 2022-01-03?
Response: What was the revenue for 2022-01-03? The revenue for 2022-01-03 was 81434000000.0 USD.0 USD.0.0.3.0’s USD.0.4.’s 36016000000.0 USD.0.0’s. Here's a summary of the news: No news content was available.. Apple, Inc. AAPL shares soared, closing on Friday, with its shares gaining 20% and doubling its

Prompt: What was the sentiment on AAPL during April 2022?
Response: What was the sentiment on AAPL during April 